# Fine-tune BlazeFace on webcam data (people split train / val / test)

`finetune_blazeface_70_20_10.ipynb` taught BlazeFace short-range the WIDER FACE + FDDB faces. This notebook continues
from its best weights with frames recorded the way Knowing Eye sees an examinee: one person at a laptop webcam,
looking down, turning, in dim light, with a second person, an empty seat, ... (`scripts/capture_webcam_faces.py`,
labelled by hand with `scripts/label_webcam_faces.py`).

**Data.** `label_webcam_faces.py --pack` put every recorded person, as a whole, into one split - their frames are
never shared between splits, because frames a second apart are near-copies:

| Data | Used for |
|---|---|
| WIDER FACE + FDDB train (2,809 images) **+ webcam train people** | fitting the weights - WIDER/FDDB stays in so general face detection is not forgotten |
| **webcam val people** | picking the best epoch and early stopping (seen indirectly - **not** reported as accuracy) |
| WIDER FACE + FDDB val | watched every epoch, to see whether general detection gets worse |
| WIDER FACE + FDDB test | scored **once** at the end (section 8) |
| **webcam test people** | never seen here - `eval_webcam_test.ipynb` scores every model on them, once |

**What changes from the 70/20/10 run.** Training starts from its `best.weights.h5` instead of Google's model, with a
lower learning rate (refining, not relearning). Half the time a webcam frame is shown as the server receives it -
scaled to 480 px wide and JPEG quality 60 (`use-monitoring.ts`) - because the fine-tuned detector ships both in the
browser (full camera frames) and inside the server's `face_landmarker.task` (those smaller JPEGs). Everything else -
the Keras rebuild of Google's model, MediaPipe-identical pre/post-processing, anchor matching, loss, landmark
distillation from the frozen original, metrics, export - is copied unchanged from `finetune_blazeface_70_20_10.ipynb`.

**Before running**
1. On Google Drive (root): `face_split_70_20_10.zip` (as for the other notebooks) and `webcam_train.zip`
   (`python scripts/label_webcam_faces.py --pack`).
2. `finetune_blazeface_70_20_10.ipynb` must have run: this starts from `knowing-eye-blazeface-70-20-10/blazeface/best.weights.h5`.
3. Runtime -> Change runtime type -> **T4 GPU**, then Runtime -> Run all. Training saves to Drive after every epoch;
   if the session dies, Run all again and it resumes.

**Rule:** do not change any setting after looking at test results and then re-test. The test sets are only honest if
they are used once.

In [ ]:
import os, sys, subprocess

# oneDNN's CPU graph fusions crash on this model inside tf.function (seen on TF 2.21). GPU runs are unaffected;
# this only matters on a CPU runtime, and must be set before TensorFlow is imported.
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # TensorFlow/Keras are preinstalled on Colab; MediaPipe is not. Its Python API needs the EGL/GLES libraries.
    subprocess.run('apt-get -qq install -y libegl1 libgles2 > /dev/null', shell=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mediapipe==1.1.1'], check=True)

import hashlib, json, math, random, re, shutil, time, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import keras
import mediapipe as mp
import numpy as np
import pandas as pd
import tensorflow as tf
from mediapipe.tasks.python import BaseOptions, vision
from tensorflow.lite.python import schema_py_generated as schema
from tensorflow.lite.tools import flatbuffer_utils

try:   # tf.lite.Interpreter is being moved to the ai_edge_litert package
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    Interpreter = tf.lite.Interpreter
trapezoid = getattr(np, 'trapezoid', None) or np.trapz   # NumPy 2 renamed trapz

for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)   # leave GPU memory for PyTorch (YOLO comparison)
print('TensorFlow', tf.__version__, '| Keras', keras.__version__, '| MediaPipe', mp.__version__)
print('GPU:', tf.config.list_physical_devices('GPU') or 'none - running on CPU')

In [ ]:
DATASET_ZIP = '/content/drive/MyDrive/face_split_70_20_10.zip'
WEBCAM_ZIP = '/content/drive/MyDrive/webcam_train.zip'
DATA_ROOT = '/content/data'
BLAZE_DIR = '/content/drive/MyDrive/knowing-eye-blazeface-70-20-10'   # finetune_blazeface_70_20_10.ipynb: the start
OUT_DIR = '/content/drive/MyDrive/knowing-eye-blazeface-webcam'       # everything that must survive the session
EXPECTED_COUNTS = {'train': 2809, 'val': 803, 'test': 402}

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

DATA_DIR = f'{DATA_ROOT}/face_split_70_20_10'
WEBCAM_DIR = f'{DATA_ROOT}/webcam_train'
if not Path(DATA_DIR, 'images').exists():
    zipfile.ZipFile(DATASET_ZIP).extractall(DATA_ROOT)   # local disk: much faster than reading from Drive
if not Path(WEBCAM_DIR, 'manifest.csv').exists():
    zipfile.ZipFile(WEBCAM_ZIP).extractall(DATA_ROOT)
counts = {s: len(os.listdir(f'{DATA_DIR}/images/{s}')) for s in ('train', 'val', 'test')}
assert counts == EXPECTED_COUNTS, 'Unexpected split sizes - wrong zip?'


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


WEBCAM_FINGERPRINT = sha256(f'{WEBCAM_DIR}/manifest.csv')[:12]
webcam = pd.read_csv(f'{WEBCAM_DIR}/manifest.csv', dtype=str).astype({'faces': int, 'excluded': int})
webcam = webcam[webcam['excluded'] == 0].reset_index(drop=True)
assert set(webcam['split']) <= {'train', 'val'}, 'webcam_train.zip must hold train / val people only'
assert (webcam['split'] == 'val').any(), 'no webcam val people - record more people and pack again'
for row in webcam.itertuples():   # every frame and label must be exactly what was labelled and packed
    stem = Path(row.file).stem
    assert sha256(f'{WEBCAM_DIR}/images/{row.file}') == row.image_sha256, f'{row.file} differs from the manifest'
    assert sha256(f'{WEBCAM_DIR}/server/images/{stem}.jpg') == row.server_sha256, f'{row.file}: server view changed'
    assert sha256(f'{WEBCAM_DIR}/labels/{stem}.txt') == row.label_sha256, f'{row.file}: label changed'
print(counts)
print(f'webcam_train.zip {WEBCAM_FINGERPRINT}: ' + ', '.join(
    f'{s} {g.participant.nunique()} people / {len(g)} frames / {g.faces.sum()} faces'
    for s, g in webcam.groupby('split')))

In [ ]:
# ---- Config ----
EPOCHS = 40
BATCH = 32
LR = 3e-4          # Adam peak learning rate (70/20/10 run: 1e-3) - refining an already fine-tuned model
PATIENCE = 10      # early stopping on the WEBCAM VAL people
SEED = 42
TRAIN_EVAL_N = 800 # fixed, un-augmented training images scored after every epoch ("training accuracy")
SERVER_VIEW_P = 0.5   # chance a webcam training frame is shown as the server receives it (480 px, JPEG q60)

# Same reporting settings as the 70/20/10 run.
IN_RANGE_MIN_FACE = 0.10
DEPLOY_CONF = 0.5

FIG_DIR = f'{OUT_DIR}/figures'
os.makedirs(FIG_DIR, exist_ok=True)
keras.utils.set_random_seed(SEED)

# Google's pretrained models - byte-identical to the copies in the repository. The original is the frozen teacher
# for the landmarks and the template for export, exactly as in the 70/20/10 run.
MODELS = {
    'blaze_face_short_range.tflite': (
        'https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/latest/'
        'blaze_face_short_range.tflite',
        'b4578f35940bf5a1a655214a1cce5cab13eba73c1297cd78e1a04c2380b0152f'),
    'face_landmarker.task': (
        'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/'
        'face_landmarker.task',
        '64184e229b263107bc2b804c6625db1341ff2bb731874b0bcc2fe6544e0bc9ff'),
}
for name, (url, sha) in MODELS.items():
    path = Path(OUT_DIR, 'pretrained', name)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(url, path)
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    print(f'{name}: {"OK, same as the repo copy" if digest == sha else "WARNING - differs from the repo copy"}')
BASE_TFLITE = f'{OUT_DIR}/pretrained/blaze_face_short_range.tflite'
BASE_TASK = f'{OUT_DIR}/pretrained/face_landmarker.task'
START_WEIGHTS = f'{BLAZE_DIR}/blazeface/best.weights.h5'
assert Path(START_WEIGHTS).exists(), 'run finetune_blazeface_70_20_10.ipynb first - this starts from its best weights'

## 1. BlazeFace as a trainable Keras model

Copied from `finetune_blazeface_70_20_10.ipynb`: the `.tflite` graph rebuilt layer by layer in Keras with Google's
weights (the frozen `teacher`), and the export back to a MediaPipe-compatible `.tflite` / `.task`.

In [ ]:
INPUT = 128
OP_NAMES = {v: k for k, v in vars(schema.BuiltinOperator).items() if not k.startswith('_')}
NP_TYPES = {schema.TensorType.FLOAT32: np.float32, schema.TensorType.FLOAT16: np.float16,
            schema.TensorType.INT32: np.int32}


@keras.saving.register_keras_serializable('blazeface')
class ChannelPad(keras.layers.Layer):
    """Zero-pads the channel axis so a residual branch matches a wider block (TFLite PAD op)."""

    def __init__(self, pad, **kwargs):
        super().__init__(**kwargs)
        self.pad = pad

    def call(self, x):
        return keras.ops.pad(x, [[0, 0], [0, 0], [0, 0], [0, self.pad]])

    def get_config(self):
        return {**super().get_config(), 'pad': self.pad}


def blazeface_from_tflite(path, batch_size=None):
    """Rebuild the .tflite graph op by op as a Keras model carrying the pretrained weights."""
    fb = flatbuffer_utils.read_model(path)
    g = fb.subgraphs[0]
    source = {}  # DEQUANTIZE output tensor -> its float16 constant

    def const(i):
        t = g.tensors[source.get(i, i)]
        return np.frombuffer(bytes(fb.buffers[t.buffer].data), NP_TYPES[t.type]).reshape(t.shape)

    def name_of(i):
        return re.sub(r'\W', '_', g.tensors[i].name.decode()).replace('__xeno_compat__1', '')

    def padding(opt):
        return 'same' if opt.padding == schema.Padding.SAME else 'valid'

    inp = keras.Input(tuple(g.tensors[g.inputs[0]].shape[1:]), batch_size=batch_size, name='input')
    x = {g.inputs[0]: inp}
    weights = []
    for op in g.operators:
        code = fb.operatorCodes[op.opcodeIndex]
        kind = OP_NAMES[max(code.builtinCode, code.deprecatedBuiltinCode)]
        ins, out, opt = list(op.inputs), op.outputs[0], op.builtinOptions
        name = name_of(out)
        if kind == 'DEQUANTIZE':
            source[out] = ins[0]
            continue
        if kind in ('CONV_2D', 'DEPTHWISE_CONV_2D', 'ADD'):
            assert opt.fusedActivationFunction == schema.ActivationFunctionType.NONE, name
        if kind == 'CONV_2D':
            k, b = const(ins[1]).astype(np.float32), const(ins[2]).astype(np.float32)
            layer = keras.layers.Conv2D(k.shape[0], k.shape[1:3], strides=(opt.strideH, opt.strideW),
                                        padding=padding(opt), name=name)
            weights.append((layer, [k.transpose(1, 2, 3, 0), b]))
            ins = ins[:1]
        elif kind == 'DEPTHWISE_CONV_2D':
            k, b = const(ins[1]).astype(np.float32), const(ins[2]).astype(np.float32)
            layer = keras.layers.DepthwiseConv2D(k.shape[1:3], strides=(opt.strideH, opt.strideW),
                                                 padding=padding(opt), name=name)
            weights.append((layer, [k.reshape(k.shape[1], k.shape[2], -1, 1), b]))
            ins = ins[:1]
        elif kind == 'RELU':
            layer = keras.layers.ReLU(name=name)
        elif kind == 'ADD':
            layer = keras.layers.Add(name=name)
        elif kind == 'PAD':
            pads = const(ins[1])
            assert not pads[:3].any() and pads[3, 0] == 0, f'{name}: only channel padding is supported'
            layer, ins = ChannelPad(int(pads[3, 1]), name=name), ins[:1]
        elif kind == 'MAX_POOL_2D':
            layer = keras.layers.MaxPooling2D((opt.filterHeight, opt.filterWidth), (opt.strideH, opt.strideW),
                                              padding=padding(opt), name=name)
        elif kind == 'RESHAPE':
            layer, ins = keras.layers.Reshape(tuple(g.tensors[out].shape[1:]), name=name), ins[:1]
        elif kind == 'CONCATENATION':
            layer = keras.layers.Concatenate(axis=opt.axis, name=name)
        else:
            raise NotImplementedError(f'TFLite op {kind} ({name})')
        args = [x[i] for i in ins]
        x[out] = layer(args if len(args) > 1 else args[0])
    model = keras.Model(inp, [x[i] for i in g.outputs], name='blazeface_short_range')
    for layer, w in weights:
        layer.set_weights(w)
    return model


def export_tflite(model, out_path, reference=None):
    """Keras -> float16 .tflite with the reference model's I/O names, output order and MediaPipe metadata."""
    reference = reference or BASE_TFLITE
    fixed = blazeface_from_tflite(reference, batch_size=1)   # static shapes -> plain RESHAPE ops
    fixed.set_weights(model.get_weights())
    try:
        converter = tf.lite.TFLiteConverter.from_keras_model(fixed)
    except Exception:   # some TF/Keras 3 combinations: go through a SavedModel instead (same result)
        saved = f'/tmp/blazeface_export_{os.getpid()}'
        fixed.export(saved, format='tf_saved_model', verbose=False)
        converter = tf.lite.TFLiteConverter.from_saved_model(saved)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_types = [tf.float16]
    fb = flatbuffer_utils.read_model_from_bytearray(bytearray(converter.convert()))

    ref = flatbuffer_utils.read_model(reference)
    g, rg = fb.subgraphs[0], ref.subgraphs[0]
    # MediaPipe reads outputs by position: [regressors (1,896,16), classificators (1,896,1)].
    if [list(g.tensors[i].shape) for i in g.outputs] == [[1, 896, 1], [1, 896, 16]]:
        g.outputs = g.outputs[::-1]
    assert [list(g.tensors[i].shape) for i in g.outputs] == [[1, 896, 16], [1, 896, 1]]
    for mine, theirs in zip(list(g.inputs) + list(g.outputs), list(rg.inputs) + list(rg.outputs)):
        g.tensors[mine].name = rg.tensors[theirs].name
    for sig in fb.signatureDefs or []:
        sig.outputs.sort(key=lambda t: list(g.outputs).index(t.tensorIndex))
    # MediaPipe refuses a float model without input-normalisation metadata: copy Google's.
    meta = next(m for m in ref.metadata if m.name == b'TFLITE_METADATA')
    buf = schema.BufferT()
    buf.data = ref.buffers[meta.buffer].data
    fb.buffers.append(buf)
    entry = schema.MetadataT()
    entry.name, entry.buffer = b'TFLITE_METADATA', len(fb.buffers) - 1
    fb.metadata = [m for m in (fb.metadata or []) if m.name != b'TFLITE_METADATA'] + [entry]
    flatbuffer_utils.write_model(fb, out_path)
    return out_path


def repack_face_landmarker(task_in, detector_tflite, task_out):
    """Copy of face_landmarker.task with its face_detector.tflite replaced."""
    with zipfile.ZipFile(task_in) as src, zipfile.ZipFile(task_out, 'w', zipfile.ZIP_STORED) as dst:
        for item in src.infolist():
            data = Path(detector_tflite).read_bytes() if item.filename == 'face_detector.tflite' else src.read(item)
            dst.writestr(item, data)
    return task_out


teacher = blazeface_from_tflite(BASE_TFLITE)   # frozen original: baseline + landmark targets
teacher.trainable = False
print(f'BlazeFace short-range rebuilt: {len(teacher.layers)} layers, {teacher.count_params():,} parameters')

## 2. MediaPipe-identical pre- and post-processing

Copied from `finetune_blazeface_70_20_10.ipynb` (letterbox, 896 SSD anchors, decoding, weighted NMS), followed by
the same sanity checks against the original `.tflite` and MediaPipe's `FaceDetector`.

In [ ]:
# MediaPipe face_detector_graph, short-range model: SSD anchor options.
SSD = dict(num_layers=4, min_scale=0.1484375, max_scale=0.75, strides=[8, 16, 16, 16])


def ssd_anchors():
    """(896, 3) [x_centre, y_centre, nominal_scale] in MediaPipe's anchor order.

    MediaPipe decodes with fixed_anchor_size=True (anchor w = h = 1); the nominal scale is only used for
    deciding which anchors are responsible for which face size during training.
    """
    n, lo, hi, strides = SSD['num_layers'], SSD['min_scale'], SSD['max_scale'], SSD['strides']
    scale = lambda i: lo + (hi - lo) * i / (n - 1)
    anchors, layer = [], 0
    while layer < n:
        scales, last = [], layer
        while last < n and strides[last] == strides[layer]:
            nxt = 1.0 if last == n - 1 else scale(last + 1)
            scales += [scale(last), math.sqrt(scale(last) * nxt)]
            last += 1
        fm = math.ceil(INPUT / strides[layer])
        anchors += [((x + 0.5) / fm, (y + 0.5) / fm, s) for y in range(fm) for x in range(fm) for s in scales]
        layer = last
    return np.array(anchors, np.float32)


ANCHORS = ssd_anchors()
assert len(ANCHORS) == 896


def letterbox(img_rgb, size=INPUT):
    """MediaPipe ImageToTensor(keep_aspect_ratio, BORDER_ZERO): centred square ROI, one bilinear warpAffine.

    Returns the uint8 input and (scale_x, scale_y, pad_x, pad_y) with tensor_xy = pad + image_xy * scale.
    """
    h, w = img_rgb.shape[:2]
    s = max(h, w) / size
    m = np.array([[1 / s, 0, (max(h, w) - w) / 2 / s], [0, 1 / s, (max(h, w) - h) / 2 / s]], np.float32)
    out = cv2.warpAffine(img_rgb, m, (size, size), flags=cv2.INTER_LINEAR,
                         borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    return out, (w / s / size, h / s / size, m[0, 2] / size, m[1, 2] / size)


def normalize(batch_uint8):
    return batch_uint8.astype(np.float32) / 127.5 - 1.0


def decode(regressors, classificators):
    """Raw (896,16), (896,1) -> boxes xyxy (input-normalised), keypoints (896,6,2), scores."""
    r = regressors / INPUT
    cx, cy = ANCHORS[:, 0] + r[:, 0], ANCHORS[:, 1] + r[:, 1]
    boxes = np.stack([cx - r[:, 2] / 2, cy - r[:, 3] / 2, cx + r[:, 2] / 2, cy + r[:, 3] / 2], 1)
    kps = r[:, 4:].reshape(-1, 6, 2) + ANCHORS[:, None, :2]
    scores = 1 / (1 + np.exp(-np.clip(classificators[:, 0].astype(np.float64), -100, 100)))
    return boxes, kps, scores


def iou_matrix(a, b):
    ix1 = np.maximum(a[:, None, 0], b[None, :, 0]); iy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(a[:, None, 2], b[None, :, 2]); iy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    area = lambda x: np.clip(x[:, 2] - x[:, 0], 0, None) * np.clip(x[:, 3] - x[:, 1], 0, None)
    union = area(a)[:, None] + area(b)[None, :] - inter
    return np.where(union > 0, inter / np.maximum(union, 1e-12), 0.0)


def weighted_nms(boxes, kps, scores, min_score, iou_thresh=0.3):
    """MediaPipe NonMaxSuppressionCalculator, algorithm WEIGHTED, overlap IoU."""
    order = np.argsort(-scores)
    order = order[scores[order] >= min_score]
    out_b, out_k, out_s = [], [], []
    while order.size:
        sim = iou_matrix(boxes[order[:1]], boxes[order])[0]
        group = order[sim > iou_thresh]
        group = group if group.size else order[:1]
        w = scores[group][:, None]
        out_b.append((boxes[group] * w).sum(0) / w.sum())
        out_k.append((kps[group] * w[:, :, None]).sum(0) / w.sum())
        out_s.append(scores[order[0]])
        if not (sim > iou_thresh).any():
            break   # MediaPipe stops when an iteration removes nothing
        order = order[sim <= iou_thresh]
    if not out_s:
        return np.zeros((0, 4), np.float32), np.zeros((0, 6, 2), np.float32), np.zeros(0, np.float32)
    return np.array(out_b, np.float32), np.array(out_k, np.float32), np.array(out_s, np.float32)


def unletterbox(boxes, lb):
    sx, sy, px, py = lb
    return (boxes - [px, py, px, py]) / [sx, sy, sx, sy]


def to_tensor_space(boxes, lb):
    sx, sy, px, py = lb
    return boxes * [sx, sy, sx, sy] + [px, py, px, py]


def predict(model, x_uint8, batch=256):
    regs, clss = [], []
    for i in range(0, len(x_uint8), batch):
        r, c = model(normalize(x_uint8[i:i + batch]), training=False)
        regs.append(r.numpy()); clss.append(c.numpy())
    return np.concatenate(regs), np.concatenate(clss)


def detect(reg, cls, lbs, min_score=0.01, iou_thresh=0.3, top_k=300):
    """Per image: decode + weighted NMS, back to image-normalised xyxy, clipped to the image."""
    out = []
    for r, c, lb in zip(reg, cls, lbs):
        boxes, kps, scores = decode(r, c)
        # MediaPipe drops boxes with non-positive width/height before NMS.
        scores = np.where((boxes[:, 2] > boxes[:, 0]) & (boxes[:, 3] > boxes[:, 1]), scores, -1.0)
        keep = np.argsort(-scores)[:top_k]
        keep = keep[scores[keep] >= min_score]
        b, _, s = weighted_nms(boxes[keep], kps[keep], scores[keep], min_score, iou_thresh)
        out.append((np.clip(unletterbox(b, lb), 0, 1), s))
    return out

In [ ]:
# Sanity checks (validation images only - the test split stays untouched until the end).
_val_paths = sorted(str(p) for p in Path(DATA_DIR, 'images', 'val').iterdir())[:25]

# (a) Keras rebuild == original .tflite
interp = Interpreter(BASE_TFLITE)
interp.allocate_tensors()
worst = 0.0
for p in _val_paths[:5]:
    x = normalize(letterbox(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB))[0][None])
    interp.set_tensor(interp.get_input_details()[0]['index'], x)
    interp.invoke()
    ref = [interp.get_tensor(d['index']) for d in interp.get_output_details()]
    ours = [t.numpy() for t in teacher(x, training=False)]
    worst = max(worst, *(np.abs(a - b).max() / np.abs(a).max() for a, b in zip(ref, ours)))
print(f'(a) Keras rebuild vs original .tflite: max relative difference {worst:.1e}')
assert worst < 1e-4


# (b) our letterbox + decode + weighted NMS == MediaPipe FaceDetector
def mediapipe_detect(model_path, paths, min_conf=DEPLOY_CONF):
    det = vision.FaceDetector.create_from_options(vision.FaceDetectorOptions(
        base_options=BaseOptions(model_asset_path=model_path), min_detection_confidence=min_conf))
    out = []
    for p in paths:
        rgb = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
        h, w = rgb.shape[:2]
        res = det.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
        out.append(np.array([[d.bounding_box.origin_x / w, d.bounding_box.origin_y / h,
                              (d.bounding_box.origin_x + d.bounding_box.width) / w,
                              (d.bounding_box.origin_y + d.bounding_box.height) / h,
                              d.categories[0].score] for d in res.detections]).reshape(-1, 5))
    det.close()
    return out


def compare_with_mediapipe(model, model_path, paths):
    """Returns (#MediaPipe detections, #ours, mean |score difference|, mean IoU) over matched faces."""
    mp_dets = mediapipe_detect(model_path, paths)
    x, lbs = zip(*(letterbox(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)) for p in paths))
    ours = detect(*predict(model, np.stack(x)), lbs, min_score=DEPLOY_CONF)
    n_mp = n_ours = 0
    dscore, ious = [], []
    for m, (b, s) in zip(mp_dets, ours):
        n_mp, n_ours = n_mp + len(m), n_ours + len(b)
        if len(m) and len(b):
            iou = iou_matrix(np.clip(m[:, :4], 0, 1), b)
            for i, j in enumerate(iou.argmax(1)):
                dscore.append(abs(m[i, 4] - s[j])); ious.append(iou[i, j])
    return n_mp, n_ours, float(np.mean(dscore)) if dscore else 0.0, float(np.mean(ious)) if ious else 1.0


n_mp, n_ours, dscore, miou = compare_with_mediapipe(teacher, BASE_TFLITE, _val_paths)
print(f'(b) MediaPipe FaceDetector vs this notebook on {len(_val_paths)} images: {n_mp} vs {n_ours} faces, '
      f'mean |score diff| {dscore:.4f}, mean IoU {miou:.3f}')

## 3. Data

* **Training**: every WIDER FACE / FDDB training image plus every frame of the webcam **train** people, decoded once
  and kept in RAM (longer side 448 px). Webcam frames are kept in both views - as the camera delivers them and as the
  server receives them - and each time one is drawn, the server view is used with probability `SERVER_VIEW_P`. Then
  the same augmentation as the 70/20/10 run: random zoom/shift crop, flip, colour jitter.
* **Validation**: the webcam **val** people (camera view) pick the epoch; WIDER FACE / FDDB val is scored alongside.
  Both are letterboxed once, exactly as MediaPipe would, and never augmented.

In [ ]:
IMAGE_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
CACHE_SIDE = 448
ZOOM = (0.8, 2.5)


def read_yolo_labels(path):
    """YOLO 'cls cx cy w h' (normalised) -> (n, 4) xyxy normalised to the image."""
    rows = [l.split() for l in Path(path).read_text().splitlines() if l.strip()] if Path(path).exists() else []
    a = np.array([[float(v) for v in r[1:5]] for r in rows], np.float32).reshape(-1, 4)
    return np.stack([a[:, 0] - a[:, 2] / 2, a[:, 1] - a[:, 3] / 2, a[:, 0] + a[:, 2] / 2, a[:, 1] + a[:, 3] / 2], 1)


def list_split(split):
    images = sorted(p for p in Path(DATA_DIR, 'images', split).iterdir() if p.suffix.lower() in IMAGE_EXT)
    return [(str(p), read_yolo_labels(Path(DATA_DIR, 'labels', split, p.stem + '.txt'))) for p in images]


def webcam_items(split):
    """(paths, boxes) per frame: [camera view, server view] - the labels are the same for both."""
    rows = webcam[webcam['split'] == split]
    return [([f'{WEBCAM_DIR}/images/{f}', f'{WEBCAM_DIR}/server/images/{Path(f).stem}.jpg'],
             read_yolo_labels(f'{WEBCAM_DIR}/labels/{Path(f).stem}.txt')) for f in rows['file']]


def read_rgb(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise IOError(f'cannot read {path}')
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def load_train_cache(items):
    """items: (paths, boxes) -> ([one cached image per path], boxes)."""
    def shrink(img):
        r = CACHE_SIDE / max(img.shape[:2])
        if r < 1:
            img = cv2.resize(img, (round(img.shape[1] * r), round(img.shape[0] * r)), interpolation=cv2.INTER_AREA)
        return img

    def load(item):
        return [shrink(read_rgb(p)) for p in item[0]], item[1]
    with ThreadPoolExecutor(8) as ex:
        return list(ex.map(load, items))


def load_eval_set(items):
    def load(item):
        lb, info = letterbox(read_rgb(item[0]))
        return lb, info
    with ThreadPoolExecutor(8) as ex:
        out = list(ex.map(load, items))
    return dict(x=np.stack([o[0] for o in out]), lb=[o[1] for o in out], gt=[i[1] for i in items],
                paths=[i[0] for i in items])


def augment(img, boxes, rng):
    """Random zoom/shift crop + flip (one warpAffine, like MediaPipe) + colour jitter.

    Returns the 128x128 uint8 input, the faces to learn and the faces to ignore (cut off by the crop),
    both as xyxy in 0..1 input coordinates.
    """
    h, w = img.shape[:2]
    roi = max(h, w) / math.exp(rng.uniform(*np.log(ZOOM)))
    lo_x, hi_x = sorted((roi / 2, w - roi / 2))
    lo_y, hi_y = sorted((roi / 2, h - roi / 2))
    cx, cy = rng.uniform(lo_x, hi_x), rng.uniform(lo_y, hi_y)
    s = roi / INPUT
    m = np.array([[1 / s, 0, -(cx - roi / 2) / s], [0, 1 / s, -(cy - roi / 2) / s]], np.float32)
    if rng.random() < 0.5:
        m[0] = [-m[0, 0], 0, INPUT - m[0, 2]]   # horizontal flip
    out = cv2.warpAffine(img, m, (INPUT, INPUT), flags=cv2.INTER_LINEAR,
                         borderMode=cv2.BORDER_CONSTANT, borderValue=0)

    b = boxes * [w, h, w, h]
    xs = (b[:, [0, 2]] * m[0, 0] + m[0, 2]) / INPUT
    ys = (b[:, [1, 3]] * m[1, 1] + m[1, 2]) / INPUT
    full = np.stack([xs.min(1), ys.min(1), xs.max(1), ys.max(1)], 1)
    clipped = np.clip(full, 0, 1)
    area = lambda a: np.clip(a[:, 2] - a[:, 0], 0, None) * np.clip(a[:, 3] - a[:, 1], 0, None)
    visible = area(clipped) / np.maximum(area(full), 1e-9)
    faces, cut = clipped[visible >= 0.6], clipped[(visible > 0) & (visible < 0.6)]

    f = out.astype(np.float32)
    f = (f - f.mean()) * rng.uniform(0.7, 1.3) + f.mean() * rng.uniform(0.7, 1.3)   # contrast, brightness
    grey = f.mean(axis=2, keepdims=True)
    f = grey + (f - grey) * (0.0 if rng.random() < 0.05 else rng.uniform(0.6, 1.4))  # saturation / greyscale
    return np.clip(f, 0, 255).astype(np.uint8), faces, cut


t0 = time.time()
train_items = [([p], b) for p, b in list_split('train')] + webcam_items('train')
train_cache = load_train_cache(train_items)
val_set = load_eval_set([(paths[0], b) for paths, b in webcam_items('val')])   # picks the epoch
wider_val_set = load_eval_set(list_split('val'))                               # watched alongside
te_idx = sorted(np.random.default_rng(SEED).permutation(len(train_items))[:TRAIN_EVAL_N])
train_eval_set = load_eval_set([(train_items[i][0][0], train_items[i][1]) for i in te_idx])
n_webcam = sum(len(views) > 1 for views, _ in train_cache)
print(f'loaded in {time.time() - t0:.0f}s: {len(train_cache)} training images ({len(train_cache) - n_webcam} WIDER/FDDB '
      f'+ {n_webcam} webcam), {len(val_set["x"])} webcam val, {len(wider_val_set["x"])} WIDER/FDDB val, '
      f'{len(train_eval_set["x"])} train-eval images')

## 4. Training targets, loss and metrics

Copied from `finetune_blazeface_70_20_10.ipynb`: SSD anchor matching (face anchor IoU >= 0.4, background < 0.3,
faces under 6% of the frame ignored), BCE with 3:1 hard-negative mining, GIoU box loss, landmark distillation from
the frozen original, and Ultralytics-identical metrics. The only change is in `make_batch`, which picks a webcam
frame's view.

In [ ]:
POS_IOU, NEG_IOU = 0.4, 0.3
MIN_TRAIN_FACE = 0.06
NOMINAL = np.stack([ANCHORS[:, 0] - ANCHORS[:, 2] / 2, ANCHORS[:, 1] - ANCHORS[:, 2] / 2,
                    ANCHORS[:, 0] + ANCHORS[:, 2] / 2, ANCHORS[:, 1] + ANCHORS[:, 2] / 2], 1)
HALF_STRIDE = np.where(np.arange(896) < 512, 0.5 / 16, 0.5 / 8).astype(np.float32)


def encode(faces, ignore):
    """Per-anchor label (1 face, 0 background, -1 ignored) and its target box (xyxy, input-normalised)."""
    faces = faces.astype(np.float32).reshape(-1, 4)
    ignore = ignore.astype(np.float32).reshape(-1, 4)
    size = np.sqrt(np.clip(faces[:, 2] - faces[:, 0], 0, None) * np.clip(faces[:, 3] - faces[:, 1], 0, None))
    ignore = np.concatenate([ignore, faces[size < MIN_TRAIN_FACE]])
    faces = faces[size >= MIN_TRAIN_FACE]
    labels = np.zeros(896, np.float32)
    targets = np.zeros((896, 4), np.float32)
    if len(faces):
        iou = iou_matrix(NOMINAL, faces)
        best_face, best_iou = iou.argmax(1), iou.max(1)
        labels[best_iou >= NEG_IOU] = -1
        labels[best_iou >= POS_IOU] = 1
        forced = iou.argmax(0)
        labels[forced] = 1
        best_face[forced] = np.arange(len(faces))
        targets = faces[best_face]
    if len(ignore):
        # Small faces can be narrower than a grid cell: widen each ignore region by half the anchor's stride,
        # otherwise no anchor centre lands inside it and the face would be taught as background.
        c, pad = ANCHORS[:, :2], HALF_STRIDE[:, None]
        inside = ((c[:, None, 0] >= ignore[None, :, 0] - pad) & (c[:, None, 0] <= ignore[None, :, 2] + pad) &
                  (c[:, None, 1] >= ignore[None, :, 1] - pad) & (c[:, None, 1] <= ignore[None, :, 3] + pad)).any(1)
        labels[inside & (labels == 0)] = -1
    return labels, targets


def make_batch(idx, seed):
    rng = np.random.default_rng(seed)
    xs, ls, ts = [], [], []
    for i in idx:
        views, boxes = train_cache[i]
        img = views[1] if len(views) > 1 and rng.random() < SERVER_VIEW_P else views[0]
        x, faces, cut = augment(img, boxes, rng)
        lab, tgt = encode(faces, cut)
        xs.append(x); ls.append(lab); ts.append(tgt)
    return np.stack(xs), np.stack(ls), np.stack(ts)


def eval_targets(ev):
    out = [encode(to_tensor_space(g, lb), np.zeros((0, 4))) for g, lb in zip(ev['gt'], ev['lb'])]
    return np.stack([o[0] for o in out]), np.stack([o[1] for o in out])


val_set['labels'], val_set['targets'] = eval_targets(val_set)
_lab = val_set['labels']
print(f'webcam val targets: {(_lab == 1).sum(1).mean():.1f} face anchors / image, '
      f'{(_lab == -1).sum(1).mean():.1f} ignored, {(_lab == 0).sum(1).mean():.0f} background')

In [ ]:
BOX_WEIGHT, KP_WEIGHT = 2.0, 20.0
NEG_POS_RATIO, MIN_NEG = 3, 16
ANCHOR_XY = tf.constant(ANCHORS[:, :2])


def decode_tf(reg):
    r = reg / INPUT
    c = ANCHOR_XY + r[..., :2]
    return tf.concat([c - r[..., 2:4] / 2, c + r[..., 2:4] / 2], -1)


def giou(a, b):
    ix1, iy1 = tf.maximum(a[..., 0], b[..., 0]), tf.maximum(a[..., 1], b[..., 1])
    ix2, iy2 = tf.minimum(a[..., 2], b[..., 2]), tf.minimum(a[..., 3], b[..., 3])
    inter = tf.nn.relu(ix2 - ix1) * tf.nn.relu(iy2 - iy1)
    area = lambda x: tf.nn.relu(x[..., 2] - x[..., 0]) * tf.nn.relu(x[..., 3] - x[..., 1])
    union = area(a) + area(b) - inter + 1e-9
    enclose = ((tf.maximum(a[..., 2], b[..., 2]) - tf.minimum(a[..., 0], b[..., 0])) *
               (tf.maximum(a[..., 3], b[..., 3]) - tf.minimum(a[..., 1], b[..., 1])) + 1e-9)
    return inter / union - (enclose - union) / enclose


def compute_losses(reg, cls, labels, targets, t_reg, t_cls):
    logits = cls[..., 0]
    pos = tf.cast(labels > 0.5, tf.float32)
    neg = tf.cast(tf.equal(labels, 0.0), tf.float32)
    n_pos = tf.reduce_sum(pos, 1)
    bce = tf.nn.sigmoid_cross_entropy_with_logits(labels=pos, logits=logits)
    neg_loss = bce * neg
    k = tf.cast(tf.minimum(tf.maximum(NEG_POS_RATIO * n_pos, MIN_NEG), tf.reduce_sum(neg, 1)), tf.int32)
    thresh = tf.gather(tf.sort(neg_loss, axis=1, direction='DESCENDING'), tf.maximum(k - 1, 0), batch_dims=1)
    hard_neg = neg * tf.cast(neg_loss >= thresh[:, None], tf.float32) * tf.cast(k[:, None] > 0, tf.float32)
    total_pos = tf.maximum(tf.reduce_sum(n_pos), 1.0)
    cls_loss = tf.reduce_sum(bce * (pos + hard_neg)) / total_pos

    box_loss = BOX_WEIGHT * tf.reduce_sum((1.0 - giou(decode_tf(reg), targets)) * pos) / total_pos

    teacher_face = tf.cast(tf.sigmoid(t_cls[..., 0]) > 0.5, tf.float32)
    kp_err = tf.reduce_mean(tf.abs(reg[..., 4:] - t_reg[..., 4:]), -1) / INPUT
    kp_loss = KP_WEIGHT * tf.reduce_sum(kp_err * teacher_face) / tf.maximum(tf.reduce_sum(teacher_face), 1.0)
    return cls_loss + box_loss + kp_loss, cls_loss, box_loss, kp_loss

In [ ]:
IOU_T = np.linspace(0.5, 0.95, 10)


def match_predictions(gt, det):
    """TP matrix (n_det, 10) - same algorithm as ultralytics BaseValidator.match_predictions."""
    tp = np.zeros((len(det), len(IOU_T)), bool)
    if len(gt) == 0 or len(det) == 0:
        return tp
    iou = iou_matrix(gt, det)
    for i, t in enumerate(IOU_T):
        m = np.nonzero(iou >= t)
        if m[0].size:
            mm = np.stack([m[0], m[1], iou[m]], 1)
            if m[0].size > 1:
                mm = mm[mm[:, 2].argsort()[::-1]]
                mm = mm[np.unique(mm[:, 1], return_index=True)[1]]
                mm = mm[np.unique(mm[:, 0], return_index=True)[1]]
            tp[mm[:, 1].astype(int), i] = True
    return tp


def smooth(y, f=0.1):
    nf = round(len(y) * f * 2) // 2 + 1
    p = np.ones(nf // 2)
    return np.convolve(np.concatenate((p * y[0], y, p * y[-1])), np.ones(nf) / nf, mode='valid')


def compute_ap(recall, precision):
    mrec = np.concatenate(([0.0], recall, [recall[-1] if len(recall) else 1.0], [1.0]))
    mpre = np.flip(np.maximum.accumulate(np.flip(np.concatenate(([1.0], precision, [0.0], [0.0])))))
    x = np.linspace(0, 1, 101)
    return trapezoid(np.interp(x, mrec, mpre), x)


def face_size(gt, lb):
    """Face size as a fraction of the image's longer side (= of the 128-px input)."""
    t = to_tensor_space(gt, lb)
    return np.sqrt(np.clip(t[:, 2] - t[:, 0], 0, None) * np.clip(t[:, 3] - t[:, 1], 0, None))


def evaluate(dets, gts, lbs, min_face=0.0):
    """min_face > 0: faces smaller than that fraction of the image are 'ignore' regions (neither TP nor FN,
    and detections on them are not FP)."""
    tps, confs, n_gt = [], [], 0
    for (boxes, scores), g, lb in zip(dets, gts, lbs):
        if min_face > 0 and len(g):
            size = face_size(g, lb)
            small, g = g[size < min_face], g[size >= min_face]
            if len(small) and len(boxes):
                hit = match_predictions(g, boxes)[:, 0] if len(g) else np.zeros(len(boxes), bool)
                keep = ~((iou_matrix(boxes, small).max(1) >= 0.5) & ~hit)
                boxes, scores = boxes[keep], scores[keep]
        n_gt += len(g)
        tps.append(match_predictions(g, boxes)); confs.append(scores)
    tp, conf = np.concatenate(tps), np.concatenate(confs)
    order = np.argsort(-conf)
    tp, conf = tp[order], conf[order]
    tpc, fpc = tp.cumsum(0), (1 - tp).cumsum(0)
    recall, precision = tpc / (n_gt + 1e-16), tpc / (tpc + fpc + 1e-16)
    ap = np.array([compute_ap(recall[:, j], precision[:, j]) for j in range(10)]) if len(tp) else np.zeros(10)
    px = np.linspace(0, 1, 1000)
    p_curve = np.interp(-px, -conf, precision[:, 0], left=1) if len(tp) else np.ones(1000)
    r_curve = np.interp(-px, -conf, recall[:, 0], left=0) if len(tp) else np.zeros(1000)
    i = smooth(2 * p_curve * r_curve / (p_curve + r_curve + 1e-16)).argmax()
    p, r = p_curve[i], r_curve[i]
    tp_n = round(r * n_gt)
    fp_n = round(tp_n / (p + 1e-16) - tp_n)
    d = conf >= DEPLOY_CONF
    tp_d = int(tp[d, 0].sum())
    return dict(precision=p, recall=r, f1=2 * p * r / (p + r + 1e-16), accuracy=p * r / (p + r - p * r + 1e-16),
                map50=ap[0], map=ap.mean(), best_conf=px[i], TP=tp_n, FP=fp_n, FN=n_gt - tp_n, faces=n_gt,
                deploy_precision=tp_d / max(int(d.sum()), 1), deploy_recall=tp_d / max(n_gt, 1),
                deploy_TP=tp_d, deploy_FP=int(d.sum()) - tp_d, deploy_FN=n_gt - tp_d)


def score(model, ev, min_faces=(0.0,)):
    dets = detect(*predict(model, ev['x']), ev['lb'])
    return [evaluate(dets, ev['gt'], ev['lb'], m) for m in min_faces]

## 5. Train

Adam, warm-up then cosine decay, batch 32, gradient clipping - as in the 70/20/10 run, from its best weights and
with a lower peak learning rate. After every epoch the notebook records the training losses, the validation loss and
metrics on the **webcam val people**, the metrics on WIDER FACE / FDDB val (all faces and webcam-range faces), and the
metrics on 800 fixed, un-augmented training images. The epoch with the best webcam-val mAP@0.5:0.95 is kept as
`best.weights.h5`; training stops after 10 epochs without improvement. **Epoch 0 is the starting model** (the
70/20/10 fine-tune) - the baseline.

Everything is written to Drive after each epoch, so a disconnected session resumes on the next Run all.

In [ ]:
student = blazeface_from_tflite(BASE_TFLITE)
student.load_weights(START_WEIGHTS)
steps = math.ceil(len(train_cache) / BATCH)
warmup = min(3 * steps, EPOCHS * steps // 10)
schedule = keras.optimizers.schedules.CosineDecay(LR * 0.1, EPOCHS * steps - warmup, alpha=0.01,
                                                  warmup_target=LR, warmup_steps=warmup)
optimizer = keras.optimizers.Adam(schedule)
optimizer.build(student.trainable_variables)


@tf.function(reduce_retracing=True)
def train_step(x_uint8, labels, targets):
    x = tf.cast(x_uint8, tf.float32) / 127.5 - 1.0
    t_reg, t_cls = teacher(x, training=False)
    with tf.GradientTape() as tape:
        reg, cls = student(x, training=True)
        losses = compute_losses(reg, cls, labels, targets, t_reg, t_cls)
    grads = tape.gradient(losses[0], student.trainable_variables)
    grads, _ = tf.clip_by_global_norm(grads, 10.0)
    optimizer.apply_gradients(zip(grads, student.trainable_variables))
    return losses


@tf.function(reduce_retracing=True)
def loss_step(x_uint8, labels, targets):
    x = tf.cast(x_uint8, tf.float32) / 127.5 - 1.0
    t_reg, t_cls = teacher(x, training=False)
    reg, cls = student(x, training=False)
    return compute_losses(reg, cls, labels, targets, t_reg, t_cls)


def validation_loss():
    total, n = np.zeros(4), 0
    for i in range(0, len(val_set['x']), 128):
        out = loss_step(val_set['x'][i:i + 128], val_set['labels'][i:i + 128], val_set['targets'][i:i + 128])
        total += np.array([float(v) for v in out]) * len(val_set['x'][i:i + 128])
        n += len(val_set['x'][i:i + 128])
    return total / n


def epoch_row(epoch, train_losses, lr):
    vl = validation_loss()
    (v_all,) = score(student, val_set)
    w_all, w_in = score(student, wider_val_set, (0.0, IN_RANGE_MIN_FACE))
    (t_all,) = score(student, train_eval_set)
    row = dict(epoch=epoch, lr=lr)
    row.update(zip(['train_loss', 'train_cls_loss', 'train_box_loss', 'train_kp_loss'], train_losses))
    row.update(zip(['val_loss', 'val_cls_loss', 'val_box_loss', 'val_kp_loss'], vl))
    row.update({f'val_{k}': v for k, v in v_all.items()})              # webcam val people
    row.update({f'wider_{k}': v for k, v in w_all.items()})            # WIDER FACE / FDDB val, all faces
    row.update({f'widerrange_{k}': v for k, v in w_in.items()})        # ... webcam-range faces
    row.update({f'train_{k}': v for k, v in t_all.items()})
    return row


run_dir = Path(OUT_DIR, 'blazeface')
run_dir.mkdir(parents=True, exist_ok=True)
hist_path, state_path = run_dir / 'history.csv', run_dir / 'state.json'
if state_path.exists():   # resume an interrupted run
    state = json.loads(state_path.read_text())
    history = pd.read_csv(hist_path).to_dict('records')
    student.load_weights(str(run_dir / 'last.weights.h5'))
    optimizer.iterations.assign(state['epoch'] * steps)
    print(f'resuming after epoch {state["epoch"]} (best so far: epoch {state["best_epoch"]})')
else:
    state = dict(epoch=0, best_epoch=0, best_fitness=-1.0, done=False, webcam_fingerprint=WEBCAM_FINGERPRINT)
    history = [epoch_row(0, [math.nan] * 4, 0.0)]
    state['best_fitness'] = history[0]['val_map']
    student.save_weights(str(run_dir / 'best.weights.h5'))
assert state.get('webcam_fingerprint') == WEBCAM_FINGERPRINT, \
    'this run was started on a different webcam_train.zip - use a new OUT_DIR for new data'


def log(row, secs):
    print(f"epoch {row['epoch']:3d} | loss {row['train_loss']:.3f} val {row['val_loss']:.3f} | "
          f"WEBCAM VAL P {row['val_precision']:.3f} R {row['val_recall']:.3f} mAP50 {row['val_map50']:.3f} "
          f"mAP50-95 {row['val_map']:.3f} | WIDER val mAP50 {row['wider_map50']:.3f} "
          f"(webcam-range {row['widerrange_map50']:.3f}) | train mAP50 {row['train_map50']:.3f} | {secs:.0f}s")


if state['epoch'] == 0:
    log(history[0], 0)
with ThreadPoolExecutor(4) as pool:
    for epoch in range(state['epoch'] + 1, EPOCHS + 1):
        if state['done']:
            break
        t0 = time.time()
        order = np.random.default_rng(SEED + epoch).permutation(len(train_cache))
        batches = pool.map(make_batch, [order[s * BATCH:(s + 1) * BATCH] for s in range(steps)],
                           [SEED * 100_000 + epoch * 1_000 + s for s in range(steps)])
        sums = np.zeros(4)
        for x, labels, targets in batches:
            sums += np.array([float(v) for v in train_step(x, labels, targets)])
        row = epoch_row(epoch, sums / steps, float(schedule(optimizer.iterations)))
        history.append(row)
        student.save_weights(str(run_dir / 'last.weights.h5'))
        if row['val_map'] > state['best_fitness']:
            state.update(best_fitness=row['val_map'], best_epoch=epoch)
            student.save_weights(str(run_dir / 'best.weights.h5'))
        state['epoch'] = epoch
        if epoch - state['best_epoch'] >= PATIENCE:
            state['done'] = True
            print(f'early stop: no webcam-val mAP@0.5:0.95 improvement for {PATIENCE} epochs')
        pd.DataFrame(history).to_csv(hist_path, index=False)
        state_path.write_text(json.dumps(state))
        log(row, time.time() - t0)

print(f"best epoch: {state['best_epoch']} (webcam val mAP@0.5:0.95 = {state['best_fitness'] * 100:.2f}%"
      + (', i.e. the starting model - fine-tuning did not improve on it)' if state['best_epoch'] == 0 else ')'))

## 6. Export for MediaPipe

* `blaze_face_short_range_webcam.tflite` - drop-in replacement for
  `frontend/public/models/blaze_face_short_range.tflite` (browser overlay).
* `face_landmarker_webcam.task` - Google's `face_landmarker.task` with this detector inside; drop-in replacement for
  `backend/ai/models/face_landmarker.task` (server pipeline).

Both are loaded back into MediaPipe on WIDER FACE / FDDB val images to prove they work. Speed is unchanged - same
network, same size - so the 70/20/10 speed numbers still apply.

In [ ]:
best = blazeface_from_tflite(BASE_TFLITE)
best.load_weights(str(run_dir / 'best.weights.h5'))
history_df = pd.DataFrame(history)
best_epoch = int(state['best_epoch'])

import contextlib, io
with contextlib.redirect_stdout(io.StringIO()):   # the converter prints every captured variable
    FT_TFLITE = export_tflite(best, f'{OUT_DIR}/blaze_face_short_range_webcam.tflite')
FT_TASK = repack_face_landmarker(BASE_TASK, FT_TFLITE, f'{OUT_DIR}/face_landmarker_webcam.task')
print(f'exported {FT_TFLITE} ({os.path.getsize(FT_TFLITE) / 1024:.0f} KB)')

n_mp, n_ours, dscore, miou = compare_with_mediapipe(best, FT_TFLITE, _val_paths)
print(f'MediaPipe FaceDetector + exported .tflite vs Keras model: {n_mp} vs {n_ours} faces, '
      f'mean |score diff| {dscore:.4f}, mean IoU {miou:.3f}')
assert abs(n_mp - n_ours) <= max(1, n_ours // 20) and miou > 0.9, 'exported model does not match - do not ship it'


def face_mesh_found(task_path):
    lm = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=task_path), num_faces=1))
    n = sum(bool(lm.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=read_rgb(p))).face_landmarks)
            for p in _val_paths)
    lm.close()
    return n


print(f'FaceLandmarker - face mesh found on {len(_val_paths)} val images: original task '
      f'{face_mesh_found(BASE_TASK)}, webcam-tuned task {face_mesh_found(FT_TASK)}')

## 7. Training curves

Epoch 0 is the starting model. The webcam-val curves are what picked the kept epoch (dotted line); the WIDER FACE /
FDDB val curve shows whether general face detection suffered. All are **validation** numbers - the test numbers come
from section 8 and from `eval_webcam_test.ipynb`.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# Palette and look matched to the paper's existing figures.
METRIC_COLORS = {'mAP@0.5': '#1F4E8C', 'Precision': '#2E8B57', 'F1-Score': '#D98C00', 'Recall': '#C0504D',
                 'mAP@0.5:0.95': '#6A3D9A', 'Accuracy': '#2B8C8C'}
plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 11, 'axes.titleweight': 'bold', 'axes.titlesize': 13,
    'axes.labelweight': 'bold', 'axes.labelsize': 12, 'figure.facecolor': 'white', 'axes.facecolor': 'white',
})


def style(ax, xlabel=None, ylabel=None, title=None):
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_linewidth(1.6)
    ax.grid(axis='y', color='#DDDDDD', linewidth=1)
    ax.set_axisbelow(True)
    if xlabel:
        ax.set_xlabel(xlabel)
        if 'Epoch' in xlabel:
            ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    if ylabel:
        ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title)


def legend_top(ax, ncol):
    ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.01), ncol=ncol, frameon=False,
              prop={'weight': 'bold', 'size': 10})


def mark_best(ax, epoch=None):
    epoch = best_epoch if epoch is None else epoch
    ax.axvline(epoch, color='#5B7A99', linestyle=':', linewidth=1.5)
    ax.annotate(f'best: epoch {epoch}', xy=(epoch, 1), xycoords=('data', 'axes fraction'), xytext=(0, 4),
                textcoords='offset points', ha='center', va='bottom', color='#1F4E8C', fontsize=9,
                fontweight='bold')


def save(fig, fname):
    fig.savefig(f'{FIG_DIR}/{fname}', dpi=200, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)


h = history_df.copy()
for col in [c for c in h.columns if c.startswith(('val_', 'wider_', 'widerrange_', 'train_'))
            and c.split('_', 1)[1] in ('precision', 'recall', 'f1', 'accuracy', 'map50', 'map')]:
    h[col] = h[col] * 100
h.round(4).to_csv(f'{run_dir}/history_all_epochs.csv', index=False)

# ---------- Validation metrics across epochs: webcam val people (solid) and WIDER FACE / FDDB val (dashed) ----------
fig, ax = plt.subplots(figsize=(13, 5.5))
for key, label in [('val_map50', 'mAP@0.5'), ('val_map', 'mAP@0.5:0.95'), ('val_precision', 'Precision'),
                   ('val_recall', 'Recall')]:
    ax.plot(h['epoch'], h[key], color=METRIC_COLORS[label], lw=2.2, label=f'{label} (webcam val)')
ax.plot(h['epoch'], h['wider_map50'], color=METRIC_COLORS['mAP@0.5'], lw=1.8, ls='--',
        label='mAP@0.5 (WIDER/FDDB val)')
style(ax, 'Training Epoch', 'Score (%)')
ax.set_ylim(0, 100)
ax.set_xlim(0, h['epoch'].max())
mark_best(ax)
legend_top(ax, ncol=3)
save(fig, 'blazeface_webcam_val_metrics_all_epochs.png')

# ---------- Training loss components across epochs ----------
fig, ax = plt.subplots(figsize=(13, 5.5))
for key, label, c in [('train_cls_loss', 'Class Loss', '#C0504D'), ('train_box_loss', 'Box (GIoU) Loss', '#D98C00'),
                      ('train_kp_loss', 'Landmark Loss', '#6A3D9A')]:
    ax.plot(h['epoch'], h[key], color=c, lw=2.2, label=label)
style(ax, 'Training Epoch', 'Loss')
ax.set_ylim(bottom=0)
ax.set_xlim(1, max(h['epoch'].max(), 2))
legend_top(ax, ncol=3)
save(fig, 'blazeface_webcam_loss_components.png')

keep = sorted({0, best_epoch, int(h['epoch'].max())})
cols = {'epoch': 'Epoch', 'val_map50': 'Webcam val mAP@0.5 (%)', 'val_map': 'Webcam val mAP@0.5:0.95 (%)',
        'val_precision': 'Webcam val Precision (%)', 'val_recall': 'Webcam val Recall (%)',
        'wider_map50': 'WIDER/FDDB val mAP@0.5 (%)', 'widerrange_map50': 'WIDER/FDDB val webcam-range mAP@0.5 (%)',
        'train_loss': 'Train Loss', 'val_loss': 'Val Loss'}
table = h[h['epoch'].isin(keep)][list(cols)].rename(columns=cols).round(2)
table['Note'] = table['Epoch'].map(lambda e: 'kept (best webcam-val mAP@0.5:0.95)' if e == best_epoch else
                                   'starting model' if e == 0 else 'last epoch')
table.to_csv(f'{run_dir}/history_checkpoints.csv', index=False)
table

## 8. WIDER FACE / FDDB test split - run once

Did adapting to webcams cost general face detection? The webcam-tuned model is scored once on the same 402 test
images as every other model, with the same code, under both protocols. The starting model's numbers are read from
`finetune_blazeface_70_20_10.ipynb`'s results, not recomputed. The result is saved; running this cell again loads it.

In [ ]:
WIDER_RESULTS = Path(OUT_DIR, 'test_results_blazeface_webcam_wider_fddb.csv')
if WIDER_RESULTS.exists():
    wider_test_df = pd.read_csv(WIDER_RESULTS)
    print(f'Already scored once - loaded {WIDER_RESULTS.name}')
else:
    test_set = load_eval_set(list_split('test'))
    reg, cls = predict(best, test_set['x'])
    dets = detect(reg, cls, test_set['lb'])
    confident = np.concatenate([s for _, s in detect(reg, cls, test_set['lb'], min_score=0.25)])
    rows = []
    for protocol, min_face in (('all faces', 0.0), ('webcam-range faces', IN_RANGE_MIN_FACE)):
        m = evaluate(dets, test_set['gt'], test_set['lb'], min_face)
        row = {'model': 'blazeface-webcam', 'protocol': protocol,
               'mAP50 (%)': m['map50'] * 100, 'mAP50-95 (%)': m['map'] * 100,
               'Precision (%)': m['precision'] * 100, 'Recall (%)': m['recall'] * 100,
               'F1 (%)': m['f1'] * 100, 'Accuracy (%)': m['accuracy'] * 100,
               'TP': m['TP'], 'FP': m['FP'], 'FN': m['FN'], 'Faces': m['faces'],
               'Precision @0.5 (%)': m['deploy_precision'] * 100, 'Recall @0.5 (%)': m['deploy_recall'] * 100,
               'TP @0.5': m['deploy_TP'], 'FP @0.5': m['deploy_FP'], 'FN @0.5': m['deploy_FN']}
        if protocol == 'all faces':
            row['Mean Confidence (%)'] = float(confident.mean() * 100) if len(confident) else 0.0
            row['Detections (n)'] = int(len(confident))
        rows.append(row)
    wider_test_df = pd.DataFrame(rows).round(2)
    wider_test_df.to_csv(WIDER_RESULTS, index=False)

previous = Path(BLAZE_DIR, 'test_results_blazeface_70_20_10.csv')
compare = pd.concat([pd.read_csv(previous), wider_test_df]) if previous.exists() else wider_test_df
cols = ['model', 'protocol', 'mAP50 (%)', 'mAP50-95 (%)', 'Precision (%)', 'Recall (%)', 'F1 (%)', 'Accuracy (%)']
print('WIDER FACE / FDDB TEST split (402 images):')
compare[cols].sort_values(['protocol', 'model']).reset_index(drop=True)

## 9. What to download and what comes next

Everything is in `MyDrive/knowing-eye-blazeface-webcam/`:

* `blaze_face_short_range_webcam.tflite` - copy over `frontend/public/models/blaze_face_short_range.tflite`
* `face_landmarker_webcam.task` - copy over `backend/ai/models/face_landmarker.task` (only the face *detector*
  inside changes - landmarks and blendshapes stay Google's)
* `test_results_blazeface_webcam_wider_fddb.csv` - the WIDER FACE / FDDB test numbers (section 8)
* `blazeface/history_all_epochs.csv`, `history_checkpoints.csv`, `figures/` - the training history
* `blazeface/best.weights.h5` - the Keras weights

**Next:** run `eval_webcam_test.ipynb`. It finds this model on Drive and scores it on the webcam **test** people
together with YOLOv8n, YOLO11n, RT-DETR-l and both earlier BlazeFace models - once.

**For the paper's methodology:** BlazeFace short-range was further fine-tuned from the 70/20/10 weights on the WIDER
FACE + FDDB training split plus *N* webcam frames from *P* training participants (participant-disjoint from the
validation and test participants), with each webcam frame shown at random either as captured or as the server
receives it (480 px wide, JPEG quality 60): Adam (peak LR 3e-4, warm-up + cosine), batch 32, up to 40 epochs with
early stopping (patience 10) on the validation participants' mAP@0.5:0.95, otherwise the same augmentation, loss and
landmark distillation as before.